In [ ]:
import pandas as pd

In [ ]:
# read in the sample for class sessions as a basis for all information for class sessions
class_sessions_df = pd.read_parquet('Sample/sample_classes.parquet')
sample_aps_df = pd.read_parquet('Sample/sample_aps.parquet')
sample_classrooms_df = pd.read_parquet('Sample/sample_classrooms.parquet')


In [ ]:
#print(class_sessions_df.head())
print(class_sessions_df.dtypes)
print(sample_aps_df.dtypes)
print(sample_classrooms_df.dtypes)

In [ ]:
#Adding additional columns to the DataFrame
# Ensure string columns use Pandas' dedicated string dtype
class_sessions_df['group'] = pd.Series(dtype="string")

# Set numerical columns with nullable integer dtype (Int64)
class_sessions_df['number_aps'] = pd.Series(dtype="Int64")
class_sessions_df['number_radios'] = pd.Series(dtype="Int64")
class_sessions_df['number_beacons'] = pd.Series(dtype="Int64")
class_sessions_df['max_stations'] = pd.Series(dtype="Int64")
class_sessions_df['min_stations'] = pd.Series(dtype="Int64")
class_sessions_df['average_cu'] = pd.Series(dtype="Int64")
class_sessions_df['median_cu'] = pd.Series(dtype="Int64")

# Compute class_duration in SECONDS (float since it involves time differences)
class_sessions_df['class_duration'] = (class_sessions_df['end_time'] - class_sessions_df['start_time']).dt.total_seconds().astype("Float64")

# Ensure duration_75 and duration_50 are also float types but nullable
class_sessions_df['duration_75'] = pd.Series(dtype="Float64")
class_sessions_df['duration_50'] = pd.Series(dtype="Float64")


In [ ]:

# Display the updated DataFrame
print(class_sessions_df.head())
print(class_sessions_df.dtypes)


In [ ]:
# Count occurrences of each Building-Room in sample_aps_df
aps_count = sample_aps_df['Building-Room'].value_counts()

# Map these counts to the number_aps column in class_sessions_df
class_sessions_df['number_aps'] = class_sessions_df['location'].map(aps_count).astype("Int64")


In [ ]:
# Create a mapping from sample_classrooms_df: Location -> Group
location_to_group = sample_classrooms_df.set_index('Location')['Group']

# Map the Group values to class_sessions_df based on location
class_sessions_df['group'] = class_sessions_df['location'].map(location_to_group).astype("string")


In [ ]:
# iterate through each location in sample_classes
# find all of the APs for the location
# iterate through each of the AP dataframe files and create a new column "room_stations" that contains a sum of the total_scount for that radio combined with all others
# use the closest beacon found for other APs
# need to identify the APs that do not have any captures for the class and remove it from the sample